In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
nl = pd.read_csv('dataset/task2_ennl_all.csv')
zh = pd.read_csv('dataset/task2_enzh_all.csv')
zh.columns = nl.columns
df=pd.concat([nl,zh], axis=0)
df.Label1.replace({'Not acceptable (Error))': 'Not acceptable (Error)'}, inplace = True)

/var/folders/2n/4742tn7s13l5fcnstlm_d7g40000gn/T/ipykernel_71494/1554644040.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.Label1.replace({'Not acceptable (Error))': 'Not acceptable (Error)'}, inplace = True)


In [3]:
import re
def zng(paragraph):
    for sent in re.findall(u'[^!?。\.\!\?]+[!?。\.\!\?]?', paragraph, flags=re.U):
        yield sent

In [5]:
df["model_name"] = df.model.apply(lambda x: " ".join(x.split()[:-1]))
source = df[["source", "book"]].drop_duplicates()
source["n_source"] = source.source.apply(lambda x: len(x.split()))
source.groupby("book")["n_source"].sum().mean()

217.75

In [6]:
import unicodedata

def remove_punctuation(text: str) -> str:
    """Remove all Unicode punctuation (including Chinese) from text."""
    return "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "P"
    )

def cal_n(row):
    if row.pair == "en-nl":
        return len(row["Translation"].split())
    elif row.pair == "en-zh":
        return len(remove_punctuation(row["Translation"]))
df["n_target"] = df.apply(cal_n, axis = 1)
dd = df.groupby(["book", "source", "pair"], as_index = False).n_target.mean()
dd1 = dd.groupby(["book", "pair"], as_index=False).n_target.sum()
dd1.groupby("pair").n_target.mean()

pair
en-nl    219.281250
en-zh    377.902083
Name: n_target, dtype: float64

In [7]:
df["Level of acceptability"] = df["Level of acceptability"].apply(lambda x: x.lower())
df["Level of creativity"] = df["Level of creativity"].apply(lambda x: x.lower())
df["Level of deviation"] = df["Level of deviation"].apply(lambda x: x.lower())


In [8]:
def score_map(row, weight):
    acc_l = row["Level of acceptability"]
    crea_l = row["Level of creativity"]
    if row["Label1"] == "Creative Shift":
        score = 1*weight[crea_l]*weight[acc_l]
    elif "acceptable" in row["Label1"]:
        score=-1
    else:
        score=0
    return score
weight = {"low":1, "medium": 1, "high": 1}
df["score_orig"] = df.apply(lambda x: score_map(x, weight), axis =1)

In [9]:
df.groupby(["pair","model"], as_index = False)[["score_orig"]].mean()

,pair,model,score_orig
0,en-nl,Claude 3.7 Sonnet,0.071429
1,en-nl,GPT-4o,-0.083333
2,en-nl,Gemini 2.5 Flash Preview,0.071429
3,en-nl,Grok3 mini,-0.086207
4,en-nl,Llama 3.3 70B Instruct,-0.304348
5,en-nl,Qwen2.5 7B Instruct,-0.439024
6,en-nl,human,0.226562
7,en-zh,Claude 3.7 Sonnet,-0.222222
8,en-zh,GPT-4o,-0.130435
9,en-zh,Gemini 2.5 Flash Preview,-0.050000


In [10]:
order = ["low", "medium", "high"]

df["Level of creativity"] = pd.Categorical(
    df["Level of creativity"],
    categories=order,
    ordered=True
)
df["Level of acceptability"] = pd.Categorical(
    df["Level of acceptability"],
    categories=order,
    ordered=True
)

In [11]:
df["model_"] = df.model.apply(lambda x: "human" if x=="human" else "LLM")

In [12]:
lang = "en-zh"
df_human = df[df.pair == lang].set_index("model_").loc["human"].dropna(subset=["Level of acceptability"])
table = pd.crosstab(df_human["Level of acceptability"], df_human["Level of creativity"])/len(df_human)
print("Contingency Table:", lang)
print(table)

lang = "en-nl"
df_human = df[df.pair == lang].set_index("model_").loc["human"].dropna(subset=["Level of acceptability"])
table = pd.crosstab(df_human["Level of acceptability"], df_human["Level of creativity"])/len(df_human)
print("Contingency Table:", lang)
print(table)

Contingency Table: en-zh
Level of creativity          low    medium      high
Level of acceptability                              
low                     0.036649  0.005236  0.000000
medium                  0.115183  0.052356  0.026178
high                    0.188482  0.366492  0.209424
Contingency Table: en-nl
Level of creativity          low    medium      high
Level of acceptability                              
low                     0.109375  0.109375  0.007812
medium                  0.242188  0.171875  0.039062
high                    0.101562  0.140625  0.078125


In [13]:
lang = "en-zh"
sub = ["Claude 3.7 Sonnet", "Gemini 2.5 Flash Preview", "GPT-4o"]
df_llm = df[df.pair == lang].set_index("model").loc[sub].dropna(subset=["Level of acceptability"])
table = pd.crosstab(df_llm["Level of acceptability"], df_llm["Level of creativity"])/len(df_llm)
print("Contingency Table:", lang)
print(table)

lang = "en-nl"
sub = ["Claude 3.7 Sonnet", "Gemini 2.5 Flash Preview", "GPT-4o"]
df_llm = df[df.pair == lang].set_index("model").loc[sub].dropna(subset=["Level of acceptability"])
table = pd.crosstab(df_llm["Level of acceptability"], df_llm["Level of creativity"])/len(df_llm)
print("Contingency Table:", lang)
print(table)

Contingency Table: en-zh
Level of creativity          low    medium      high
Level of acceptability                              
low                     0.148515  0.000000  0.000000
medium                  0.188119  0.059406  0.009901
high                    0.376238  0.198020  0.019802
Contingency Table: en-nl
Level of creativity          low    medium      high
Level of acceptability                              
low                     0.072581  0.088710  0.008065
medium                  0.282258  0.137097  0.016129
high                    0.241935  0.153226  0.000000
